In [4]:
import pandas as pd
import requests
import time

def filtrar_compuestos_chembl(input_csv, output_csv):
    # 1. Leer el archivo CSV especificando el separador correcto
    print("Cargando el dataset original...")
    try:
        df = pd.read_csv(input_csv, sep=';')
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo '{input_csv}'. Asegúrate de que esté en la misma carpeta.")
        return
    
    # Limpiar espacios en los nombres de las columnas
    df.columns = df.columns.str.strip()
    
    # Identificar la columna del ID de ChEMBL
    id_column = 'Molecule ChEMBL ID'
    if id_column not in df.columns:
        print(f"Error: No se encontró la columna '{id_column}'. Columnas disponibles: {df.columns.tolist()}")
        return

    # Eliminar duplicados o nulos en los IDs para no hacer peticiones de más
    chembl_ids = df[id_column].dropna().unique().tolist()
    print(f"Se encontraron {len(chembl_ids)} compuestos únicos para analizar.")

    compuestos_filtrados = []
    
    # 2. Configurar la API de ChEMBL
    base_url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"
    
    print("Consultando la API de ChEMBL (esto puede tomar un momento dependiendo del volumen)...")
    
    for idx, chembl_id in enumerate(chembl_ids):
        # Progreso en consola
        if (idx + 1) % 10 == 0 or idx == len(chembl_ids) - 1:
            print(f"Procesando: {idx + 1}/{len(chembl_ids)}...")
            
        # Parámetros de búsqueda para la API
        params = {
            'molecule_chembl_id': chembl_id,
            'limit': 1000, # Traer suficientes registros de actividad por compuesto
            'format': 'json'
        }
        
        try:
            response = requests.get(base_url, params=params, timeout=15)
            if response.status_code == 200:
                data = response.json()
                activities = data.get('activities', [])
                
                es_valido = False
                # 3. Evaluar las actividades del compuesto
                for act in activities:
                    organismo = str(act.get('target_organism', '')).lower()
                    pref_name = str(act.get('target_pref_name', '')).lower()
                    
                    # Comprobar si cumple con las dos condiciones básicas (Plasmodium falciparum + Cysteine Protease)
                    if "plasmodium falciparum" in organismo:
                        if "cysteine protease" in pref_name or "cistein" in pref_name or "falcipain" in pref_name:
                            es_valido = True
                            break # Encontró al menos una coincidencia válida, pasamos al siguiente compuesto
                
                if es_valido:
                    compuestos_filtrados.append(chembl_id)
                    
            else:
                print(f"Advertencia: Error {response.status_code} al consultar el ID {chembl_id}")
                
        except Exception as e:
            print(f"Error de conexión con el ID {chembl_id}: {e}")
        
        # Pequeña pausa para ser amigables con el servidor de ChEMBL
        time.sleep(0.1)

    # 4. Filtrar el DataFrame original y guardar el resultado
    df_resultado = df[df[id_column].isin(compuestos_filtrados)]
    
    df_resultado.to_csv(output_csv, sep=';', index=False)
    print(f"\n¡Proceso completado con éxito!")
    print(f"Compuestos que cumplen el criterio: {len(df_resultado)}")
    print(f"Resultados guardados en: '{output_csv}'")

# --- Ejecución del Script ---
if __name__ == "__main__":
    # Cambia los nombres de los archivos si lo requieres
    archivo_entrada = 'dataset_inicial.csv'
    archivo_salida = 'dataset_filtrado_cysteine_protease.csv'
    
    filtrar_compuestos_chembl(archivo_entrada, archivo_salida)

Cargando el dataset original...
Se encontraron 537 compuestos únicos para analizar.
Consultando la API de ChEMBL (esto puede tomar un momento dependiendo del volumen)...
Procesando: 10/537...
Procesando: 20/537...
Procesando: 30/537...
Procesando: 40/537...
Procesando: 50/537...
Procesando: 60/537...
Procesando: 70/537...
Procesando: 80/537...
Procesando: 90/537...
Procesando: 100/537...
Procesando: 110/537...
Procesando: 120/537...
Procesando: 130/537...
Procesando: 140/537...
Procesando: 150/537...
Procesando: 160/537...
Procesando: 170/537...
Procesando: 180/537...
Procesando: 190/537...
Procesando: 200/537...
Procesando: 210/537...
Procesando: 220/537...
Procesando: 230/537...
Procesando: 240/537...
Procesando: 250/537...
Procesando: 260/537...
Procesando: 270/537...
Procesando: 280/537...
Procesando: 290/537...
Procesando: 300/537...
Procesando: 310/537...
Procesando: 320/537...
Procesando: 330/537...
Procesando: 340/537...
Procesando: 350/537...
Procesando: 360/537...
Procesando: